# PALSAR-2 destriping: before and after

Converts one PALSAR-2 L2.2 scene, cropped to an AOI, with and without `--destripe` for each polarization, then shows them on two maps side by side: original on the left, destriped on the right. Panning or zooming either map moves the other. Below each pair is what was removed, so you can check that slicks were left alone.

Both versions use the same stretch (taken from the original), so any difference is the destriping.

In [1]:
import sys
from pathlib import Path

# Repo root (parent of notebooks/) so `utils` and `src` import the same way the scripts do
REPO_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(REPO_ROOT))

import base64
import warnings

import numpy as np
import rasterio
from ipyleaflet import FullScreenControl, ImageOverlay, Map, basemaps
from ipywidgets import HTML, HBox, Image, Layout, VBox, jslink
from rasterio.enums import Resampling
from rasterio.errors import NotGeoreferencedWarning
from rasterio.io import MemoryFile
from rasterio.vrt import WarpedVRT
from rasterio.warp import transform_bounds

from src.alos2_convert_to_tiff import convert_scene_to_geotiff
from utils.variables import (
    ALOS2_GEOTIFF_PATH,
    JAXA_L22_BUCKET_URL,
    JAXA_L22_PREFIX,
    SEARCH_AOIS,
)

In [ ]:
# Scene + AOI pairs to try, each with the AOI fully inside the scene's swath. Pick one with EXAMPLE.
# Scene IDs are as in the granule CSVs / GeoTIFF names (ALOS2 + orbit + frame - date _ mode).
# The AOI is a name from SEARCH_AOIS, or (name, [min_lon, min_lat, max_lon, max_lat]) for a box of
# its own; the name goes in the output file names.
EXAMPLES = [
    # Western Niger Delta, near shore
    ("ALOS2656093550-260717_WBDR2.2GUD", "leaky-infrastructure-70111"),
    ("ALOS2656093500-260717_WBDR2.2GUD", "leaky-infrastructure-70111"),
    ("ALOS2660973500-260819_WBDR2.2GUD", "leaky-infrastructure-70111"),
    ("ALOS2662303500-260828_WBDR2.2GUD", "leaky-infrastructure-70111"),
    ("ALOS2667183500-260930_WBDR2.2GUD", "leaky-infrastructure-70111"),
    # Eastern Niger Delta, offshore
    ("ALOS2656093550-260717_WBDR2.2GUD", "leaky-infrastructure-59260"),
    ("ALOS2662303550-260828_WBDR2.2GUD", "leaky-infrastructure-59260"),
    # Congo coast, near Pointe-Noire (three different orbits)
    ("ALOS2651063700-260613_WBDR2.2GUD", "leaky-infrastructure-514300"),
    ("ALOS2651803700-260618_WBDR2.2GUD", "leaky-infrastructure-514300"),
    ("ALOS2655203700-260711_WBDR2.2GUD", "leaky-infrastructure-514300"),
    # South China Sea, off southern Vietnam
    ("ALOS2645553350-260507_WBDR2.2GUD", "iceye-4415276-match"),
    ("ALOS2662113350-260827_WBDR2.2GUD", "iceye-4415276-match"),
    # Same area, the frame to the south (GEE: JAXA/ALOS/PALSAR-2/Level2_2/ScanSAR/ALOS2639343400-260326_WBDR2_2GUD).
    # ~296 x 80 km centred on 110.767, 10.896: as wide as fits in the swath there (the coast, at
    # ~108.96 E, is too far west to reach without going off the scene's east edge)
    (
        "ALOS2639343400-260326_WBDR2.2GUD",
        ("iceye-4415276-south", [109.411, 10.533, 112.123, 11.258]),
    ),
]
EXAMPLE = 0
SCENE, AOI = EXAMPLES[EXAMPLE]
for i, (scene, aoi) in enumerate(EXAMPLES):
    print(
        f"{'>' if i == EXAMPLE else ' '} {i:2d}  {scene}  {aoi if isinstance(aoi, str) else aoi[0]}"
    )

# HH and/or HV; WBS/WWS scenes are HH only
POLARIZATIONS = ["HH", "HV"]
# Set BBOX = [min_lon, min_lat, max_lon, max_lat] to crop SCENE to your own box instead of AOI's
BBOX = None
# Written next to the viewer's GeoTIFFs, so the viewer picks them up too
OUT_DIR = ALOS2_GEOTIFF_PATH
# True remakes both GeoTIFFs even if they exist (needed after changing an AOI's box or
# src/alos2_destripe.py, and to time both in the comparison at the end); False reuses them
OVERWRITE = True
# Longest side of the images drawn on the maps; bigger crops are shown downsampled (the GeoTIFFs aren't)
MAX_PX = 4096
# dB stretch percentiles, from the original; lower the top one to bring out dark slicks on calm water
PCT = (2, 98)
# What was removed is shown from -REMOVED_DB (black) to +REMOVED_DB (white)
REMOVED_DB = 1.0

In [ ]:
# Each polarization twice: (original, destriped). The original reads only the crop from JAXA; the
# destriped version also reads the whole scene at 50 m, which is what the stripes are estimated from.
# Each conversion is timed for the comparison at the end (only if the file was actually written).
import time

if BBOX is not None:
    aoi_name, bbox = "destripe-demo", BBOX
elif isinstance(AOI, str):
    aoi_name, bbox = AOI, [float(v) for v in SEARCH_AOIS[AOI]["bbox"].split(",")]
else:
    aoi_name, bbox = AOI
pairs, seconds = {}, {}
for pol in POLARIZATIONS:
    url = f"{JAXA_L22_BUCKET_URL}/{JAXA_L22_PREFIX}{SCENE}_{pol}_SLP.tif"
    pairs[pol] = []
    for destripe in (False, True):
        start = time.time()
        path = convert_scene_to_geotiff(
            url,
            bbox=bbox,
            aoi_name=aoi_name,
            out_dir=OUT_DIR,
            destripe=destripe,
            overwrite=OVERWRITE,
        )
        # A file reused from an earlier run wasn't timed
        if path.stat().st_mtime >= start:
            seconds[pol, destripe] = time.time() - start
        pairs[pol].append(path)
    print(pol, *[p.name for p in pairs[pol]], sep="\n  ")

In [ ]:
def read_db(path):
    """GeoTIFF (linear gamma0) in Web Mercator as dB, at most MAX_PX on a side, with its lon/lat bounds."""
    with (
        rasterio.open(path) as src,
        WarpedVRT(src, crs="EPSG:3857", nodata=np.nan) as vrt,
    ):
        scale = max(1, max(vrt.width, vrt.height) / MAX_PX)
        shape = (int(vrt.height / scale), int(vrt.width / scale))
        # Averaged in linear power, before converting to dB
        data = vrt.read(1, out_shape=shape, resampling=Resampling.average)
        bounds = transform_bounds(vrt.crs, "EPSG:4326", *vrt.bounds)
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(data > 0, 10 * np.log10(data), np.nan), bounds


def to_png(data, vmin, vmax):
    """Grayscale PNG bytes stretched from vmin to vmax; NaN transparent."""
    valid = np.isfinite(data)
    gray = np.clip((np.where(valid, data, vmin) - vmin) / (vmax - vmin), 0, 1)
    with warnings.catch_warnings(), MemoryFile() as mem:
        warnings.simplefilter("ignore", NotGeoreferencedWarning)
        with mem.open(
            driver="PNG",
            width=data.shape[1],
            height=data.shape[0],
            count=2,
            dtype="uint8",
        ) as png:
            png.write(np.stack([gray * 255, valid * 255]).astype("uint8"))
        return mem.read()


def overlay(png, bounds, name):
    west, south, east, north = bounds
    url = "data:image/png;base64," + base64.b64encode(png).decode()
    return ImageOverlay(url=url, bounds=((south, west), (north, east)), name=name)


# Per polarization: original and destriped on two maps side by side whose pan and zoom are linked,
# then what was removed
def make_map(center, layer):
    m = Map(
        center=center,
        zoom=10,
        basemap=basemaps.Esri.WorldImagery,
        scroll_wheel_zoom=True,
        layout=Layout(height="600px", width="50%"),
    )
    m.add(layer)
    # Button at the top left; full screen shows just this map (the other still follows its pan and zoom)
    m.add(FullScreenControl())
    return m

In [ ]:
panels = []
for pol, (original_tif, destriped_tif) in pairs.items():
    original, bounds = read_db(original_tif)
    destriped, _ = read_db(destriped_tif)
    vmin, vmax = np.nanpercentile(original, PCT)
    removed = original - destriped

    west, south, east, north = bounds
    center = ((south + north) / 2, (west + east) / 2)
    left = make_map(
        center, overlay(to_png(original, vmin, vmax), bounds, f"{pol} original")
    )
    right = make_map(
        center, overlay(to_png(destriped, vmin, vmax), bounds, f"{pol} destriped")
    )
    # Moving either map moves the other
    jslink((left, "center"), (right, "center"))
    jslink((left, "zoom"), (right, "zoom"))

    panels += [
        HTML(
            f"<h3>{SCENE} {pol}</h3>original (left) | destriped (right), "
            f"stretched {vmin:.1f} to {vmax:.1f} dB"
        ),
        HBox([left, right]),
        HTML(
            f"Removed (original minus destriped), -{REMOVED_DB:g} dB black to +{REMOVED_DB:g} dB white; "
            f"99% of pixels within ±{np.nanpercentile(np.abs(removed), 99):.2f} dB"
        ),
        Image(
            value=to_png(removed, -REMOVED_DB, REMOVED_DB),
            format="png",
            layout=Layout(width="50%"),
        ),
    ]
VBox(panels)

## How long destriping takes

The two ways of getting this AOI, as timed in the conversion cell above:
- **Clip only**: read just the AOI from JAXA and convert it.
- **Clip + destripe**: read the AOI, plus the whole scene at 50 m and its `_MSK.tif` (~1 MB), estimate the stripes on the whole scene, and remove them from the AOI.

Only files written in that run are timed; with `OVERWRITE = False`, files reused from an earlier run show as not timed. Remote reads from JAXA vary from run to run, so treat the times as rough.

In [ ]:
def fmt(key):
    return f"{seconds[key]:8.1f}s" if key in seconds else f"{'not timed':>9}"


print(f"{SCENE}, crop {aoi_name}")
print(f"{'':4} {'clip only':>9} {'+ destripe':>10} {'extra':>9}")
for pol in POLARIZATIONS:
    clip, full = (pol, False), (pol, True)
    extra = (
        f"{seconds[full] - seconds[clip]:8.1f}s"
        if clip in seconds and full in seconds
        else ""
    )
    print(f"{pol:4} {fmt(clip)} {fmt(full):>10} {extra:>9}")
if len(seconds) < 2 * len(POLARIZATIONS):
    print(
        "\nSome files were reused from an earlier run; set OVERWRITE = True to time everything."
    )

## Scratch: NISAR, PALSAR-2 and Sentinel-1 for one location

Downloads every NISAR scene for a location and date range (as `nisar_access.py` would). It then shows them on one map with every PALSAR-2 ScanSAR and Sentinel-1 GRD image from Earth Engine over the same dates. Tick images on and off in the panel beside the map: NISAR layers start on, Earth Engine layers start off.

Needs `earthengine-api` (installed into the kernel by the next cell), an Earth Engine-enabled Cloud project, and `EARTHDATA_TOKEN` in `.env`. Run the import cell, the settings cell and the `read_db` / `to_png` / `overlay` cell above first; this cell reuses those helpers and `MAX_PX` / `PCT`.

In [ ]:
import sys
!uv pip install --python {sys.executable} earthengine-api

In [10]:
import argparse
from concurrent.futures import ProcessPoolExecutor, as_completed
from datetime import UTC, datetime

import ee
from ipyleaflet import LayersControl, Rectangle, TileLayer
from ipywidgets import Checkbox
from tqdm.auto import tqdm

import sys

sys.path.append("../")

from src.asf_sar_search import construct_api_search
from src.nisar_access import process_scene
from utils.generate_aoi_details import point_to_bbox
from utils.geometry import crop_pixels
from utils.variables import (
    EARTHDATA_TOKEN,
    NISAR_GCOV_PIXEL_SIZE_M,
    NISAR_GCOV_SCENE_SIZE_M,
    NISAR_GEOTIFF_PATH,
    NISAR_REMOTE_MAX_PIXELS,
)

# ---- Settings ----
# Location in the utils/aois.py format; the name goes in the output file names.
# Set LOCATION = None to use SEARCH_AOIS[LOCATION_NAME] instead.
LOCATION_NAME = "iceye-950555583"
LOCATION = None
START, END = "2025-04-01", "2025-06-01"  # YYYY-MM-DD (Earth Engine treats END as exclusive)

# NISAR, as the nisar_access.py arguments. Only GCOV/GSLC: the map shows intensity in dB.
NISAR_PRODUCT = "GCOV"
NISAR_FREQUENCY = "A"  # A: 10 m, B: 80 m
NISAR_POLARIZATIONS = ["HHHH"]  # GCOV: VVVV / VHVH for DV scenes; GSLC: HH, HV. A missing one is reported as failed
NISAR_CROP = True  # read only the location's box; False downloads whole .h5 files
NISAR_OUT_DIR = NISAR_GEOTIFF_PATH
NISAR_OVERWRITE = False
NISAR_KEEP_H5 = False
NISAR_WORKERS = 4

# Earth Engine
EE_PROJECT = "skytruth-tech"  
PALSAR_POLARIZATIONS = ["HH"]
S1_POLARIZATIONS = ["VV"]
EE_CLIP = False  # clip Earth Engine images to the location's box
# dB stretch per polarization for the Earth Engine layers
EE_DB_RANGE = {"HH": (-25, 0), "HV": (-32, -12), "VV": (-25, 0), "VH": (-30, -10)}

# ---- Location ----
aoi = SEARCH_AOIS[LOCATION_NAME] if LOCATION is None else LOCATION
if "point" in aoi:
    lon, lat = (float(v) for v in aoi["point"].split(","))
    bbox_str = point_to_bbox(lon, lat, aoi["buffer_km"])
else:
    bbox_str = aoi["bbox"]
loc_bbox = [float(v) for v in bbox_str.split(",")]

# ---- NISAR: search, then read or download and convert each scene (as nisar_access.py does) ----
if NISAR_PRODUCT not in ("GCOV", "GSLC"):
    raise ValueError("The map shows intensity in dB, so use GCOV or GSLC")
if not EARTHDATA_TOKEN:
    raise SystemExit("EARTHDATA_TOKEN is not set; add it to .env")
# Same remote-or-download choice as nisar_access.py
if NISAR_PRODUCT == "GCOV":
    pixel_m = NISAR_GCOV_PIXEL_SIZE_M[NISAR_FREQUENCY]
    band_px = (
        crop_pixels(loc_bbox, pixel_m)
        if NISAR_CROP
        else round(NISAR_GCOV_SCENE_SIZE_M / pixel_m) ** 2
    )
    remote = band_px * len(NISAR_POLARIZATIONS) <= NISAR_REMOTE_MAX_PIXELS
else:
    remote = NISAR_CROP

granules = construct_api_search(
    bbox=bbox_str,
    start=START,
    end=END,
    processing_level=NISAR_PRODUCT,
    location=LOCATION_NAME,
)
scenes = granules.unique(subset="Granule Name", keep="first", maintain_order=True)
args = argparse.Namespace(
    polarization=NISAR_POLARIZATIONS,
    frequency=NISAR_FREQUENCY,
    db=False,  # linear; read_db converts to dB for the map
    aoi=LOCATION_NAME,
    out_dir=NISAR_OUT_DIR,
    overwrite=NISAR_OVERWRITE,
    crs=None,
    keep_h5=NISAR_KEEP_H5,
)
nisar_tifs = []
# Scenes run in parallel, so the bar counts scenes done; each one's output is printed above it as it finishes
with (
    ProcessPoolExecutor(max_workers=NISAR_WORKERS) as pool,
    tqdm(
        total=scenes.height,
        desc=f"NISAR ({'remote reads' if remote else 'whole-file downloads'})",
        unit="scene",
    ) as bar,
):
    futures = {
        pool.submit(
            process_scene, name, url, remote, loc_bbox if NISAR_CROP else None, args
        ): name
        for name, url in scenes.select("Granule Name", "URL").iter_rows()
    }
    for future in as_completed(futures):
        tifs, error, output = future.result()
        bar.write(f"{futures[future]}\n{output.rstrip()}" + (f"\n  FAILED: {error}" if error else ""))
        nisar_tifs += tifs
        bar.update()
# Granule names are NISAR_L2_PR_GCOV_..._<start time>_..., so part 11 is the acquisition start
nisar_tifs.sort(key=lambda p: p.name.split("_")[11])

# ---- Earth Engine: every PALSAR-2 ScanSAR and Sentinel-1 image over the box and dates ----
ee.Initialize(project=EE_PROJECT)
region = ee.Geometry.Rectangle(loc_bbox)


shape: (0, 5)
┌──────────────┬─────────────────┬──────────────────┬─────┬───────────┐
│ Granule Name ┆ Processing Date ┆ Processing Level ┆ URL ┆ Size (MB) │
│ ---          ┆ ---             ┆ ---              ┆ --- ┆ ---       │
│ str          ┆ str             ┆ str              ┆ str ┆ str       │
╞══════════════╪═════════════════╪══════════════════╪═════╪═══════════╡
└──────────────┴─────────────────┴──────────────────┴─────┴───────────┘


NISAR (remote reads): 0scene [00:00, ?scene/s]

In [11]:

def ee_images(collection_id):
    """(image, date, bands) for each image in the collection over the region and dates, oldest first."""
    collection = (
        ee.ImageCollection(collection_id)
        .filterBounds(region)
        .filterDate(START, END)
        .sort("system:time_start")
    )
    for feature in collection.getInfo()["features"]:
        ms = feature["properties"]["system:time_start"]
        date = datetime.fromtimestamp(ms / 1000, UTC).strftime("%Y-%m-%d %H:%M")
        yield ee.Image(feature["id"]), date, {b["id"] for b in feature["bands"]}


def ee_layer(image, pol, name):
    """Map tiles of a dB image from Earth Engine, stretched for its polarization."""
    low, high = EE_DB_RANGE[pol]
    image = image.clip(region) if EE_CLIP else image
    url = image.getMapId({"min": low, "max": high})["tile_fetcher"].url_format
    return TileLayer(url=url, name=name, attribution="Google Earth Engine")


west, south, east, north = loc_bbox
m = Map(
    center=((south + north) / 2, (west + east) / 2),
    zoom=10,
    basemap=basemaps.Esri.WorldImagery,
    scroll_wheel_zoom=True,
    layout=Layout(height="800px", flex="1"),
)
# One checkbox per image, in the panel beside the map, oldest first across all three sources. Ticking adds the layer to the map and unticking
# removes it, so hidden layers don't load tiles. (ipyleaflet's visible=False only makes a layer
# transparent, and LayersControl can't undo that, so it isn't used to hide them.)
checkboxes = []  # (date, checkbox)


def add_toggle(layer, date, on):
    box = Checkbox(value=on, description=layer.name, indent=False, layout=Layout(width="auto"))
    box.observe(lambda change: m.add(layer) if change.new else m.remove(layer), "value")
    if on:
        m.add(layer)
    checkboxes.append((date, box))


counts = {"PALSAR-2": 0, "Sentinel-1": 0}
# PALSAR-2 L2.2 bands are DN; gamma0 dB = 10*log10(DN^2) - 83
for image, date, bands in ee_images("JAXA/ALOS/PALSAR-2/Level2_2/ScanSAR"):
    counts["PALSAR-2"] += 1
    for pol in [p for p in PALSAR_POLARIZATIONS if p in bands]:
        dn = image.select(pol)
        db = dn.pow(2).log10().multiply(10).subtract(83).updateMask(dn.gt(0))
        add_toggle(ee_layer(db, pol, f"{date} PALSAR-2 {pol}"), date, on=False)
# Sentinel-1 GRD is already in dB
for image, date, bands in ee_images("COPERNICUS/S1_GRD"):
    counts["Sentinel-1"] += 1
    for pol in [p for p in S1_POLARIZATIONS if p in bands]:
        add_toggle(ee_layer(image.select(pol), pol, f"{date} S1 {pol}"), date, on=False)

# NISAR GeoTIFFs as image overlays, each with its own stretch; these start switched on
for tif in tqdm(nisar_tifs):
    data, bounds = read_db(tif)
    vmin, vmax = np.nanpercentile(data, PCT)
    parts = tif.stem.split("_")
    pol = next((p for p in parts if p in NISAR_POLARIZATIONS), "")
    # Acquisition start, e.g. 20260827T001756, in the same format as the Earth Engine dates
    date = datetime.strptime(parts[11], "%Y%m%dT%H%M%S").strftime("%Y-%m-%d %H:%M")
    add_toggle(overlay(to_png(data, vmin, vmax), bounds, f"{date} NISAR {pol}"), date, on=True)

m.add(
    Rectangle(
        bounds=((south, west), (north, east)),
        color="yellow",
        weight=2,
        fill=False,
        name=LOCATION_NAME,
    )
)
# Layer list at the top right shows what's on the map now (handy in full screen, where the panel is hidden);
# full screen button at the top left
m.add(LayersControl(position="topright"))
m.add(FullScreenControl())
print(
    f"\n{LOCATION_NAME}, {START} to {END}: {len(nisar_tifs)} NISAR GeoTIFFs, "
    f"{counts['PALSAR-2']} PALSAR-2 and {counts['Sentinel-1']} Sentinel-1 images"
)
# Layers ticked later draw on top
checkboxes = [box for _, box in sorted(checkboxes, key=lambda pair: pair[0])]
HBox([m, VBox(checkboxes, layout=Layout(height="800px", width="300px", overflow_y="auto"))])

0it [00:00, ?it/s]


iceye-950555583, 2025-04-01 to 2025-06-01: 0 NISAR GeoTIFFs, 0 PALSAR-2 and 8 Sentinel-1 images
